In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_samples = 30
n_coord_genes = 5
n_noise_genes = 15

# a shared latent factor - the "true" pathway activity per patient
factor = rng.normal(size=n_samples)

# 5 genes that all follow the shared factor (+ small individual noise) -
# simulates a real, coordinated pathway
coord_data = {
    f"COORD_{i}": factor * rng.normal(1.0, 0.15) + rng.normal(scale=0.2, size=n_samples)
    for i in range(n_coord_genes)
}

# 15 genes that are pure independent noise - simulates unrelated genes,
# should NOT show strong PC1 structure
noise_data = {
    f"NOISE_{i}": rng.normal(size=n_samples)
    for i in range(n_noise_genes)
}

samples = [f"sample_{i}" for i in range(n_samples)]
expression = pd.DataFrame({**coord_data, **noise_data}, index=samples).T
expression.columns = samples

expression.shape  # (20 genes, 30 samples)

(20, 30)

In [2]:
coord_submatrix = expression.loc[[f"COORD_{i}" for i in range(5)]]
noise_submatrix = expression.loc[[f"NOISE_{i}" for i in range(5)]]

In [3]:
from romapy.core import ROMA

roma = ROMA(center="standard")

result_coord = roma._compute_module(coord_submatrix, global_center=None)
result_noise = roma._compute_module(noise_submatrix, global_center=None)

print("coordinated genes l1:", result_coord["l1"])
print("noise genes l1:", result_noise["l1"])

coordinated genes l1: 0.9664101803915424
noise genes l1: 0.32646128428595605


In [4]:
correlation = np.corrcoef(result_coord["scores"], factor)[0, 1]
print("correlation between PC1 scores and true factor:", correlation)

correlation between PC1 scores and true factor: 0.9964367593593537


In [5]:
print(type(result_coord["scores"]))     # should be pd.Series
print(result_coord["scores"].index[:3]) # should be sample names, e.g. sample_0, sample_1...
print(result_coord["gene_weights"].index.tolist())  # should be the 5 COORD_* gene names

<class 'pandas.Series'>
Index(['sample_0', 'sample_1', 'sample_2'], dtype='str')
['COORD_0', 'COORD_1', 'COORD_2', 'COORD_3', 'COORD_4']
